# Access to remote TUD data
## Mounting an SMB Share on macOS
There are two ways — through the GUI or the terminal.

### Method 1: Finder (GUI) — Easiest

In Finder, press ⌘K (or go to Go → Connect to Server). Type the server address:

   smb://tudelft.net/staff-umbrella

Click Connect. Enter your username and password when prompted. The share appears in Finder under Locations in the sidebar, and is mounted at:

   /Volumes/staff-umbrella

In [ ]:
import xarray as xr
import tarfile
import numpy as np
import matplotlib.pyplot as plt

import ok_netCDF as onc

In [ ]:
dataDir="/Volumes/staff-umbrella/PARSAX Vadatech SkyTorque Data/2026/2026-06/"
tarName="2026-06-09/level2/archive_2026-06-09T09.tar"
#dataDir="/Users/Shared/OK/Nextcloud/data/parsax/_data/" 
#tarName="archive_2026-05-27T14.tar"

fTarName=(f"{dataDir}{tarName}")


In [ ]:
index = onc.index_tar(fTarName)
#with open("tar_index.json", "w") as f:
    #json.dump(index, f)

n = 5  # Get 5th file
filename, metadata = list(index.items())[n]

print(filename)

In [ ]:
ds = onc.open_nc_from_tar(fTarName, index, filename)

In [ ]:
print("The netCDF file contents:")
with xr.set_options(display_max_rows=75):
    print(ds)


### Visualize a 2D NetCDF Variable
Select a 2D variable from `ds` and plot it using xarray and matplotlib. 

In [ ]:

two_d_vars = [v for v in ds.data_vars if ds[v].ndim == 2]
if not two_d_vars:
    raise ValueError(
        f"No 2D variable found. Variables and dims: {[(v, ds[v].dims) for v in ds.data_vars]}"
    )

print("2D variables found:", two_d_vars)


In [ ]:

var_name = two_d_vars[0]
print("Plotting 2D variable:", var_name)

fig, ax = plt.subplots(figsize=(10, 6))
ds[var_name].T.plot(ax=ax, cmap="viridis")
ax.set_ylim(0, 2500)  # set vertical axis range
ax.set_title(var_name)
plt.show()

In [ ]:
var_name='ZDR' # 'ncp_HH'
#noise_linear = ds[var_name]
#noise_db = 10 * np.log10(noise_linear)

fig, ax = plt.subplots(figsize=(10, 6))
ds[var_name].T.plot(ax=ax, cmap="viridis")
ax.set_ylim(0, 2500)  # set vertical axis range
ax.set_title(var_name)
plt.show()

In [ ]:
#var_name='noise_level_HH'
print(ds[var_name])    
if var_name not in ds.data_vars:
    raise KeyError(f"{var_name} not found in dataset variables: {list(ds.data_vars)}")

data = ds[var_name].values.flatten()
#data = ds['rawreflectivity_HH'].values.flatten()-ds['rawreflectivity_VV'].values.flatten()
print(np.mean(data), np.std(data), np.min(data), np.max(data))
data = data[np.isfinite(data)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(data, bins=60, color="steelblue", edgecolor="black")
ax.set_title(f"Histogram of {var_name}")
ax.set_xlabel(var_name)
ax.set_ylabel("Count")
plt.show()